# Phase 3B - Themes by Country Analysis

I will use NLP techniques to classify the URL-slugs for themes in a hope to get more insights into the narrative framing each country uses in their news coverage.

Example:
- https://thesil.ca/mcmaster-athletes-show-out-in-2026-milano-cortina-winter-olympics/
- https://wdet.org/2026/02/25/the-metro-team-usa-women-shine-as-americans-bring-home-33-medals-at-the-2026-winter-olympics/



In [ ]:
import sys
!{sys.executable} -m pip install -q -U google-cloud-bigquery google-cloud-bigquery-storage pyarrow db-dtypes pandas-gbq

In [ ]:
import tldextract
from urllib.parse import urlparse
from pathlib import Path
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from collections import Counter
import re

In [ ]:
# load query data from repo

data_path = Path("../data/gdelt_gkg_feb2026_relevance_ladder_lvl3_mapped.parquet")
df = pd.read_parquet(data_path, engine="pyarrow")
# df["day"] = pd.to_datetime(df["day"])

df.shape

In [ ]:
# Focus on strict tier
df_strict = df[df["rel_strict"]].copy()

In [ ]:
def clean_slug(url):
    if not isinstance(url, str):
        return ""
    
    path = urlparse(url).path.lower()
    
    # remove digits
    path = re.sub(r"\d+", " ", path)
    
    # replace hyphens & slashes with spaces
    path = path.replace("-", " ").replace("/", " ")
    
    # remove non-letters
    path = re.sub(r"[^a-z\s]", " ", path)
    
    # collapse spaces
    path = re.sub(r"\s+", " ", path).strip()
    
    return path

df_strict["slug_text"] = df_strict["url"].apply(clean_slug)
df_strict["slug_text"].head()

In [ ]:
# Remove garbled slugs
def is_garbled_slug(s: str) -> bool:
    if not isinstance(s, str):
        return True
    
    s = s.strip().lower()
    if len(s) < 25:
        return True
    
    # too few alphabetic chars relative to length
    alpha = sum(c.isalpha() for c in s)
    if alpha / max(len(s), 1) < 0.7:
        return True
    
    # too many very short tokens (random gibberish)
    toks = s.split()
    if len(toks) >= 6:
        short = sum(len(t) <= 2 for t in toks)
        if short / len(toks) > 0.5:
            return True
    
    # looks like "article xxxx yyy zzz" (template garbage)
    if re.match(r"^article\s+[a-z]{2,4}\s+[a-z]{2,4}\s+[a-z]{2,4}", s):
        return True
    
    return False

In [ ]:
# Filter out garbled slugs
df_strict["garbled"] = df_strict["slug_text"].apply(is_garbled_slug)

In [ ]:
df_strict["garbled"].mean(), df_strict.shape

In [ ]:
# only keep slugs with length >20
df_zs = df_strict[df_strict["slug_text"].str.len() > 20].copy()
df_zs.shape

In [ ]:
df_zs = df_strict[~df_strict["garbled"]].copy()
df_zs.shape

## Text Classification with NLP model

My goal is to classify the url slugs into themes to better understand what each country reports on.

Model:
- mDeBERTa-v3-base-mnli-xnli

This multilingual model can perform natural language inference (NLI) on 100 languages and is therefore also suitable for multilingual zero-shot classification.

Classification Categories (Themes):
- "medal victories and competition results",
- "athlete profile or human interest story",
- "broadcast ratings and business of the Olympics",
- "controversy, scandal, or rule violation",
- "politics or geopolitical story",
- "climate, weather, or snow conditions",
- "logistics, infrastructure, or host city preparations"

In [ ]:
import sys
!{sys.executable} -m pip install -q huggingface_hub requests

In [ ]:
#from bertopic import BERTopic
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from huggingface_hub import InferenceClient
import os, time, json, hashlib
from pathlib import Path
import pandas as pd

In [ ]:
os.environ["HF_TOKEN"] = "INSERT_HF_TOKEN"

In [ ]:
# Make sure Huggingface token is set
assert "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"], "Set os.environ['HF_TOKEN'] first"

In [ ]:
# Sample 100 articles per country
MIN_N = 100
MAX_PER_COUNTRY = 200

counts = df_zs["publisher_country_final"].value_counts()
countries = counts[counts >= MIN_N].index.tolist()

sample_df = (
    df_zs[df_zs["publisher_country_final"].isin(countries)]
    .groupby("publisher_country_final", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_PER_COUNTRY), random_state=42))
    .reset_index(drop=True)
)

sample_df.shape, sample_df[["publisher_country_final","slug_text"]].head()

In [ ]:
# use GPU if available (locally on Mac M4)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

In [ ]:
# Load model and tokenizer
model_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

In [ ]:
# MNLI label mapping
id2label = model.config.id2label
label2id = {v.lower(): k for k, v in id2label.items()}
ENTAIL_ID = label2id.get("entailment", None)

if ENTAIL_ID is None:
    # fallback: try common mapping
    # id2label: {0:'CONTRADICTION',1:'NEUTRAL',2:'ENTAILMENT'}
    ENTAIL_ID = 2

In [ ]:
LABELS = [
    "medal victories and competition results",
    "athlete profile or human interest story",
    "broadcast ratings and business of the Olympics",
    "controversy, scandal, or rule violation",
    "politics or geopolitical story",
    "climate, weather, or snow conditions",
    "logistics, infrastructure, or host city preparations"
]

In [ ]:
# zero shot classification with batching
def zero_shot_nli_batch(
    texts: list[str],
    candidate_labels: list[str],
    hypothesis_template: str = "This text is about {}.",
    batch_size: int = 16,
    max_length: int = 256,
):
    """
    Returns a list of arrays (len(texts)) where each array has shape (num_labels,)
    representing normalized entailment probabilities across labels.
    """
    all_scores = []
    N = len(texts)
    L = len(candidate_labels)

    # Pre-build hypotheses once
    hypotheses = [hypothesis_template.format(lbl) for lbl in candidate_labels]

    for start in range(0, N, batch_size):
        batch_texts = texts[start:start+batch_size]
        b = len(batch_texts)

        # Expand to b*L pairs
        premises = np.repeat(batch_texts, L).tolist()
        hyps = hypotheses * b

        enc = tokenizer(
            premises,
            hyps,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            logits = model(**enc).logits  # (b*L, 3)
            probs = torch.softmax(logits, dim=-1)[:, ENTAIL_ID]  # (b*L,)
            probs = probs.reshape(b, L).detach().cpu().numpy()

        # Normalize per text to sum to 1
        row_sums = probs.sum(axis=1, keepdims=True)
        probs = np.divide(probs, row_sums, out=np.zeros_like(probs), where=row_sums != 0)

        all_scores.append(probs)

    return np.vstack(all_scores)  # shape (N, L)

In [ ]:
# Inference
texts = sample_df["slug_text"].tolist()
scores = zero_shot_nli_batch(texts, LABELS, batch_size=16)

In [ ]:
# Build results dataframe
zs_df = sample_df[["publisher_country_final", "day", "slug_text"]].copy()

for j, lbl in enumerate(LABELS):
    zs_df[f"p__{lbl}"] = scores[:, j]

# Top label for quick checks
p_cols = [c for c in zs_df.columns if c.startswith("p__")]
zs_df["top_label"] = zs_df[p_cols].idxmax(axis=1).str.replace("p__", "", regex=False)
zs_df["top_prob"] = zs_df[p_cols].max(axis=1)

zs_df.head()

In [ ]:
# Export Dataframe with scores
out_path = Path("../data/zero_shot_mnli_results.parquet")
zs_df.to_parquet(out_path, index=False)

out_path

In [ ]:
# Exclude Japan because of too much garbled data and misclassification
zs_df = zs_df[zs_df["publisher_country_final"] != "Japan"].copy()

In [ ]:
# Country-level comparison
country_frame = (
    zs_df.groupby("publisher_country_final")[p_cols]
    .mean()
)

country_frame_pct = country_frame.div(country_frame.sum(axis=1), axis=0) * 100
country_frame_pct = country_frame_pct.rename(columns=lambda c: c.replace("p__", ""))

country_frame_pct.round(1)

In [ ]:
plot_df = country_frame_pct.loc[
    country_frame_pct.sum(axis=1).sort_values(ascending=False).index
]

plot_df.plot(kind="bar", stacked=True, figsize=(14,6))
plt.title("Zero-shot narrative framing by publisher country (MNLI, soft shares)")
plt.ylabel("% share of framing (avg entailment probs)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Frame")
plt.tight_layout()
plt.show()

In [ ]:
frame_tone = (
    zs_df
    .groupby("top_label")["top_prob"]
    .mean()
    .sort_values(ascending=False)
)

frame_tone

## Narrative Framing vs Tone

Find out what drives the tone of a news coverage.
- Are narrative framing and tone correlated?
- Does focus on reporting about medals and athletes drive positive tone?


In [ ]:
# Average tone per country
tone_country = (
    df[df["rel_strict"]]
        .groupby("publisher_country_final")["tone"]
        .mean()
)

# Merge with framing shares
analysis_df = country_frame_pct.copy()
analysis_df["avg_tone"] = tone_country

analysis_df

In [ ]:
# Correlation
corr = analysis_df.corr()

corr["avg_tone"].sort_values(ascending=False)

In [ ]:
import sys
!{sys.executable} -m pip install statsmodels

In [ ]:
# Regression: medal victories coverage on tone

import statsmodels.api as sm

X = analysis_df[["medal victories and competition results"]]
X = sm.add_constant(X)
y = analysis_df["avg_tone"]

model = sm.OLS(y, X).fit()
model.summary()

## Regression Insights

- Medal-focused framing explains approximately 60% of cross-country variation in tone during the Milano Cortina 2026 Winter Olympics.
- (+0.0764) in tone for every 1% more coverage of medals and victories!

In [ ]:
x = analysis_df["medal victories and competition results"]
y = analysis_df["avg_tone"]

# Fit line
coef = model.params
x_vals = np.linspace(x.min(), x.max(), 100)
y_vals = coef["const"] + coef["medal victories and competition results"] * x_vals

plt.figure(figsize=(8,5))

plt.scatter(x, y)

for country in analysis_df.index:
    plt.text(x.loc[country], y.loc[country], country, fontsize=8)

plt.plot(x_vals, y_vals)

plt.xlabel("% Medal Framing")
plt.ylabel("Average Tone")
plt.title("Medal Framing Explains Cross-Country Tone Variation")
plt.axhline(0, color="black", linewidth=1)
plt.text(
    0.05,
    0.95,
    f"$R^2$ = {model.rsquared:.2f}\np = {model.pvalues[1]:.3f}",
    transform=plt.gca().transAxes,
    verticalalignment='top'
)

plt.tight_layout()
plt.show()

In [ ]:
# 2 variable model

X = analysis_df[[
    "medal victories and competition results",
    "controversy, scandal, or rule violation",
]]

X = sm.add_constant(X)
y = analysis_df["avg_tone"]

model_2 = sm.OLS(y, X).fit()
model_2.summary()

## Interpretation

- 

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(
    analysis_df["medal victories and competition results"],
    analysis_df["controversy, scandal, or rule violation"],
    s=analysis_df["avg_tone"] * 200,
    alpha=0.6
)

for country in analysis_df.index:
    plt.text(
        analysis_df.loc[country, "medal victories and competition results"],
        analysis_df.loc[country, "controversy, scandal, or rule violation"],
        country,
        fontsize=8
    )

plt.xlabel("% Medal Framing")
plt.ylabel("% Controversy Framing")
plt.title("Framing Tradeoff Across Countries (Bubble ~ Tone)")
plt.tight_layout()
plt.show()